
# 实践作业：线性回归
欢迎来到你的第一个实践作业! 在这个作业中，你将用一个变量实现线性回归，以预测一家餐厅的利润。


# 一、 多变量线性回归
在这个实验室中，你将扩展数据结构和以前开发的例程以支持多个特征。几个例程的更新使得该实验室看起来很冗长，但它对以前的例程做了微小的调整，使得它可以快速回顾。


# 目录

- [1.1 目标](#11-目标)
- [1.2 工具](#12-工具)
- [1.3 记号](#13-记号)
- [2 问题陈述](#2-问题陈述)
  - [2.1 用矩阵表示训练集](#21-用矩阵表示训练集)
  - [2.2 参数向量w、b](#22-参数向量wb)
- [3 多变量的模型预测](#3-多变量的模型预测)
  - [3.1 逐元素预测](#31-逐元素预测)
  - [3.2 单一预测，向量](#32-单一预测向量)
- [4 用多个变量计算成本](#4-用多个变量计算成本)
- [5 多变量的梯度下降](#5-多变量的梯度下降)
  - [5.1 计算多变量的梯度](#51-计算多变量的梯度)
  - [5.2 多变量的梯度下降](#52-多变量的梯度下降)
- [6 恭喜你](#6-恭喜你)


## 1.1 目标
- 扩展我们的回归模型例程以支持多个特征
  - 扩展数据结构以支持多特征
  - 重写预测、成本和梯度程序以支持多个特征
  - 利用NumPy np.dot对其实现进行矢量处理，以提高速度和简单性

## 1.2 工具

- NumPy，一个用于科学计算的流行库
- Matplotlib，一个用于绘制数据的流行库

In [1]:
import copy,math
import os
import numpy as np
import matplotlib.pyplot as plt
# 使用相对路径找到style文件
current_dir = os.path.dirname(os.path.abspath('__file__'))
style_path = os.path.join(os.path.dirname(current_dir), '..', 'deeplearning.mplstyle')
plt.style.use(style_path)


## 1.3 记号

下面是你将遇到的一些记号的摘要，针对多种功能进行了更新

| 一般记号 | 描述 | Python 变量名（如果有） |
|---------|------|------------------------|
| $a$ | 标量，非粗体 | |
| $\mathbf{a}$ | 矢量，粗体 | |
| $\mathbf{A}$ | 矩阵，粗体大写 | |
| Regression | | |
| $\mathbf{X}$ | 训练实例矩阵 | X_train |
| $\mathbf{y}$ | 训练实例标签值 | y_train |
| $\mathbf{x}^{(i)}, y^{(i)}$ | 第 $i_{th}$ 个训练实例 | X[i], y[i] |
| m | 训练实例个数 | m |
| n | 每个实例中特征个数 | n |
| $\mathbf{w}$ | 参数：权重 | w |
| $b$ | 参数：偏差 | b |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | 由参数 $\mathbf{w}$, $b$ 计算出的 $\mathbf{x}^{(i)}$ 的模型预测结果：<br>$f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b$ | f_wb |



如果你想要更紧凑的表格格式：

```markdown
## 2 问题陈述

你将使用住房价格预测的例子。训练数据集包含三个例子，有四个特征（尺寸、卧室、楼层和，年龄），如下表所示。请注意，与先前的实验室不同，尺寸的单位是平方英尺而不是1000平方英尺。这导致了一个问题，你将在下一个实验中解决这个问题。

| 尺寸 (平方英尺) | 卧室数量 | 楼层数量 | 房龄 | 价格 (1000万) |
|:---:|:---:|:---:|:---:|:---:|
| 2104 | 5 | 1 | 45 | 460 |
| 1416 | 3 | 2 | 40 | 232 |
| 852 | 2 | 1 | 35 | 178 |

你将用这些信建立一个线性回归模型，这样你就可以预测其他房子的价格。例如，一个有1200平方英尺的房子，3个卧室，1层楼，40年房龄。

请运行以下代码单元来创建你的 `X_train` 和 `y_train` 变量。


In [2]:
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

## 2.1 用矩阵X表示训练集

与上面的表格类似，例子被存储在NumPy矩阵X_train中。矩阵的每一行代表一个例子。当你有 m 的训练例子（在我们的例子中m是三个），并且有n个特征（在我们的例子中是四个），X是一个尺寸为 (m, n) 的矩阵（m行，n列）。

$$
\mathbf{X} = \begin{pmatrix}
x_0^{(0)} & x_1^{(0)} & \cdots & x_{n-1}^{(0)} \\
x_0^{(1)} & x_1^{(1)} & \cdots & x_{n-1}^{(1)} \\
\cdots & \cdots & \cdots & \cdots \\
x_0^{(m-1)} & x_1^{(m-1)} & \cdots & x_{n-1}^{(m-1)}
\end{pmatrix}
$$

符号说明：

- $\mathbf{x}^{(i)}$ 是包含实例 i 的向量。$\mathbf{x}^{(i)} = (x_0^{(i)}, x_1^{(i)}, \cdots, x_{n-1}^{(i)})$
- $x_j^{(i)}$ 是实例中的第j元素。括号中的上标表示实例编号，下标表示一个元素。

### 显示输入数据


In [3]:
# 数据存储在numpy数组/矩阵中
print(f"X Shape: {X_train.shape}, X Type:{type(X_train)})")
print(X_train)
print(f"y Shape: {y_train.shape}, y Type:{type(y_train)})")
print(y_train)

X Shape: (3, 4), X Type:<class 'numpy.ndarray'>)
[[2104    5    1   45]
 [1416    3    2   40]
 [ 852    2    1   35]]
y Shape: (3,), y Type:<class 'numpy.ndarray'>)
[460 232 178]


## 2.2 参数向量w、b

- w是一个有n元素的向量。
  - 每个元素包含与一个特征相关的参数。
  - 在我们的数据集中，n是4。
  - 从概念上讲，我们把它画成一个列向量

$$
\mathbf{w} = \begin{pmatrix}
w_0 \\
w_1 \\
\cdots \\
w_{n-1}
\end{pmatrix}
$$

- b是一个标量参数

为了演示，w和b将被加载一些接近最优解的初始选择值。w是一个一维NumPy向量。

In [4]:
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])
print(f"w_init shape: {w_init.shape}, b_init type: {type(b_init)}")

w_init shape: (4,), b_init type: <class 'float'>


w 的每个元素确实对应 X 的每一列（特征）。用一维数组而不是列向量是因为：

NumPy 的广播机制会自动处理维度
代码更简洁
符合机器学习库的惯例
语义上更清晰（权重集合 vs 向量特征）
X = np.array([[2104, 5, 1, 45],
              [1416, 3, 2, 40],
              [852, 2, 1, 35]])  # (3, 4)

w = np.array([0.39, 18.75, -53.36, -26.42])  # (4,)
b = 785.18

一维数组：一行代码搞定
predictions = X @ w + b  # (3, 4) @ (4,) = (3,) ✓
结果: [预测1, 预测2, 预测3]

如果 w 是列向量 (4, 1)
w_col = w.reshape(-1, 1)
predictions = X @ w_col + b  # (3, 4) @ (4, 1) = (3, 1)
结果: [[预测1], [预测2], [预测3]] - 需要额外处理
